# LTX-Video with Pre-computed Text Embeddings and Stitching

This notebook demonstrates:
1. Encoding text prompts using the T5 text encoder
2. Running LTX-Video with pre-computed embeddings (skipping text encoder loading)
3. Generating multiple video clips
4. Stitching clips together like ltx-video-distilled-tester

## Setup

In [ ]:
# Install dependencies
!git clone https://github.com/Lightricks/LTX-Video.git
%cd LTX-Video
!pip install -e .[inference] -q

In [ ]:
# Import required libraries
import torch
import os
from pathlib import Path
from transformers import T5EncoderModel, T5Tokenizer
from inference import infer
from moviepy.editor import VideoFileClip, concatenate_videoclips
import tempfile

## Step 1: Encode Text Prompts

We'll create a function to encode prompts and save embeddings.

In [ ]:
# Text encoder configuration
TEXT_ENCODER_REPO = "PixArt-alpha/PixArt-XL-2-1024-MS"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load text encoder
print("Loading T5 text encoder...")
text_encoder = T5EncoderModel.from_pretrained(
    TEXT_ENCODER_REPO, subfolder="text_encoder"
)
tokenizer = T5Tokenizer.from_pretrained(
    TEXT_ENCODER_REPO, subfolder="tokenizer"
)

text_encoder = text_encoder.to(device)
text_encoder = text_encoder.to(torch.bfloat16)
text_encoder.eval()
print("Text encoder loaded!")

In [ ]:
def encode_and_save_prompt(prompt, negative_prompt="", max_length=256, output_path=None):
    """Encode a prompt and save embeddings to a file."""
    with torch.inference_mode():
        # Encode positive prompt
        text_inputs = tokenizer(
            prompt,
            padding="max_length",
            max_length=max_length,
            truncation=True,
            add_special_tokens=True,
            return_tensors="pt",
        )
        
        text_input_ids = text_inputs.input_ids.to(device)
        prompt_attention_mask = text_inputs.attention_mask.to(device)
        
        prompt_embeds = text_encoder(
            text_input_ids, attention_mask=prompt_attention_mask
        )[0]
        
        # Encode negative prompt
        negative_prompt_embeds = None
        negative_prompt_attention_mask = None
        if negative_prompt:
            uncond_input = tokenizer(
                negative_prompt,
                padding="max_length",
                max_length=max_length,
                truncation=True,
                return_attention_mask=True,
                add_special_tokens=True,
                return_tensors="pt",
            )
            negative_prompt_attention_mask = uncond_input.attention_mask.to(device)
            negative_prompt_embeds = text_encoder(
                uncond_input.input_ids.to(device),
                attention_mask=negative_prompt_attention_mask,
            )[0]
        
        # Create output path
        if output_path is None:
            output_dir = Path("embeddings")
            output_dir.mkdir(exist_ok=True)
            safe_name = "".join([c for c in prompt[:30] if c.isalnum() or c in (' ', '_')]).strip().replace(' ', '_')
            output_path = output_dir / f"ltx_emb_{safe_name}.pt"
        
        # Save embeddings
        embedding_data = {
            'prompt_embeds': prompt_embeds.cpu(),
            'prompt_attention_mask': prompt_attention_mask.cpu(),
            'prompt': prompt,
        }
        
        if negative_prompt_embeds is not None:
            embedding_data['negative_prompt_embeds'] = negative_prompt_embeds.cpu()
            embedding_data['negative_prompt_attention_mask'] = negative_prompt_attention_mask.cpu()
            embedding_data['negative_prompt'] = negative_prompt
        
        torch.save(embedding_data, output_path)
        print(f"Embeddings saved to {output_path}")
        return str(output_path)

## Step 2: Encode Multiple Prompts

Let's encode prompts for a sequence of video clips that we'll stitch together.

In [ ]:
# Define prompts for a sequence
prompts = [
    "A serene lake surrounded by mountains at sunset, with reflections on the water",
    "The camera slowly pans across the lake, revealing a small boat in the distance",
    "A close-up of water ripples as a fish jumps out of the water",
]

negative_prompt = "worst quality, inconsistent motion, blurry, jittery, distorted"

# Encode all prompts
embedding_paths = []
for i, prompt in enumerate(prompts):
    print(f"\nEncoding prompt {i+1}/{len(prompts)}: {prompt[:50]}...")
    emb_path = encode_and_save_prompt(prompt, negative_prompt)
    embedding_paths.append(emb_path)

print(f"\n✓ All {len(embedding_paths)} prompts encoded!")

## Step 3: Free Up VRAM

Now that we have embeddings, we can unload the text encoder to free up VRAM.

In [ ]:
# Free up text encoder memory
del text_encoder
del tokenizer
torch.cuda.empty_cache()
print("Text encoder unloaded, VRAM freed!")

## Step 4: Generate Video Clips with Pre-computed Embeddings

Now we'll generate video clips using the pre-computed embeddings, without loading the text encoder.

In [ ]:
# Configuration
config = {
    "pipeline_config": "configs/ltxv-13b-0.9.8-distilled.yaml",
    "height": 704,
    "width": 1216,
    "num_frames": 121,  # ~4 seconds at 30 fps
    "frame_rate": 30,
    "seed": 42,
    "image_cond_noise_scale": 0.15,
    "offload_to_cpu": False,
    "output_path": "output_clips",
}

# Create output directory
os.makedirs(config["output_path"], exist_ok=True)

In [ ]:
# Generate clips
video_clips = []

for i, emb_path in enumerate(embedding_paths):
    print(f"\n{'='*60}")
    print(f"Generating clip {i+1}/{len(embedding_paths)}")
    print(f"{'='*60}")
    
    # Run inference with pre-computed embeddings
    infer(
        embeddings_path=emb_path,
        prompt="",  # Not used when embeddings_path is provided
        negative_prompt="",  # Not used when embeddings_path is provided
        **config,
    )
    
    # Find the generated video
    output_dir = Path(config["output_path"])
    videos = sorted(output_dir.glob("*.mp4"), key=lambda x: x.stat().st_mtime)
    if videos:
        latest_video = str(videos[-1])
        video_clips.append(latest_video)
        print(f"✓ Clip {i+1} generated: {latest_video}")

print(f"\n✓ All {len(video_clips)} clips generated!")

## Step 5: Stitch Videos Together

Finally, we'll stitch the generated clips into a single video.

In [ ]:
def stitch_videos(video_paths, output_path="stitched_video.mp4"):
    """Stitch multiple video clips together."""
    print(f"\nStitching {len(video_paths)} clips together...")
    
    # Load video clips
    clips = [VideoFileClip(path) for path in video_paths]
    
    # Concatenate clips
    final_clip = concatenate_videoclips(clips, method="compose")
    
    # Write to file with high quality settings
    final_clip.write_videofile(
        output_path,
        codec="libx264",
        audio=False,
        threads=4,
        ffmpeg_params=["-crf", "18", "-preset", "slow"]
    )
    
    # Clean up
    for clip in clips:
        clip.close()
    
    print(f"✓ Stitched video saved to {output_path}")
    return output_path

# Stitch all clips
if len(video_clips) > 1:
    stitched_video = stitch_videos(video_clips, "final_stitched_video.mp4")
    print(f"\n🎬 Final video: {stitched_video}")
else:
    print("\n⚠ Only one clip generated, skipping stitching")

## Display Result

In [ ]:
from IPython.display import Video

if len(video_clips) > 1 and os.path.exists("final_stitched_video.mp4"):
    Video("final_stitched_video.mp4", embed=True)
elif video_clips:
    Video(video_clips[0], embed=True)

## Advanced: Image-to-Video with Stitching

You can also use the last frame of each clip as the starting image for the next clip, creating smooth transitions.

In [ ]:
import cv2
from PIL import Image

def extract_last_frame(video_path, output_image_path):
    """Extract the last frame from a video."""
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count - 1)
    ret, frame = cap.read()
    cap.release()
    
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(frame_rgb)
        image.save(output_image_path)
        return output_image_path
    return None

# Example: Generate clips with image-to-video conditioning
def generate_with_i2v_stitching(embedding_paths, config):
    """Generate videos with each using the last frame of the previous as conditioning."""
    clips = []
    conditioning_image = None
    
    for i, emb_path in enumerate(embedding_paths):
        print(f"\nGenerating clip {i+1} with embeddings...")
        
        # First clip: no conditioning, subsequent clips: use last frame
        if conditioning_image:
            infer(
                embeddings_path=emb_path,
                conditioning_media_paths=[conditioning_image],
                conditioning_start_frames=[0],
                prompt="",
                negative_prompt="",
                **config,
            )
        else:
            infer(
                embeddings_path=emb_path,
                prompt="",
                negative_prompt="",
                **config,
            )
        
        # Get the generated video
        output_dir = Path(config["output_path"])
        videos = sorted(output_dir.glob("*.mp4"), key=lambda x: x.stat().st_mtime)
        latest_video = str(videos[-1])
        clips.append(latest_video)
        
        # Extract last frame for next clip
        conditioning_image = f"frame_{i}.jpg"
        extract_last_frame(latest_video, conditioning_image)
        print(f"✓ Clip {i+1} done, extracted last frame for next clip")
    
    return clips

print("\n📝 Advanced stitching function defined.")
print("Call generate_with_i2v_stitching(embedding_paths, config) to use it.")

## Summary

This notebook demonstrates:

1. ✅ **Text Encoding**: Pre-compute embeddings using T5 encoder
2. ✅ **VRAM Optimization**: Free text encoder memory before video generation
3. ✅ **Video Generation**: Generate clips using pre-computed embeddings
4. ✅ **Video Stitching**: Combine multiple clips into one video
5. ✅ **Advanced I2V**: Use last frame as conditioning for smooth transitions

### Benefits:

- **Reduced VRAM**: Skip loading ~3GB text encoder during generation
- **Reusability**: Encode once, generate multiple times
- **Flexibility**: Run encoding and generation on different machines
- **Efficiency**: Generate long videos by stitching shorter clips
